# Cassette lower-corner YOLO Pose — Colab

Works with [Siuuuers/dl](https://github.com/Siuuuers/dl): clone + LFS, install Ultralytics, run pose inference, optional short retrain.

**Runtime → Change runtime type → GPU** (T4 is fine).

Note: live RealSense D405 viewing is local-only; Colab is for dataset/model work.


## 1. GPU check

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('Enable a GPU runtime: Runtime → Change runtime type → GPU')


## 2. Clone repo (Git LFS)

In [ ]:
import os
from pathlib import Path

REPO = 'https://github.com/Siuuuers/dl.git'
ROOT = Path('/content/dl')

!apt-get -qq update && apt-get -qq install -y git-lfs >/dev/null
!git lfs install

if ROOT.exists():
    print('Repo already present:', ROOT)
else:
    !git clone --depth 1 {REPO} {ROOT}

%cd {ROOT}
!git lfs pull
print('weights:', (ROOT / 'weights/cassette_lower_epoch173.pt').stat().st_size, 'bytes')
print('train images:', len(list((ROOT / 'data/images/train').glob('*.png'))))


## 3. Install packages

Colab already has CUDA PyTorch. We install a current Ultralytics that can load YOLO26-Pose checkpoints.


In [ ]:
%pip -q install -U ultralytics opencv-python-headless
import ultralytics, torch
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


## 4. Inference on a sample validation image

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import cv2
from IPython.display import Image, display

ROOT = Path('/content/dl')
weights = ROOT / 'weights/cassette_lower_epoch173.pt'
sample = sorted((ROOT / 'data/images/val').glob('*.png'))[0]
out_dir = ROOT / 'runs/colab_predict'
out_dir.mkdir(parents=True, exist_ok=True)

model = YOLO(str(weights))
results = model.predict(
    source=str(sample),
    imgsz=1280,
    conf=0.25,
    device=0,
    save=True,
    project=str(out_dir),
    name='preview',
    exist_ok=True,
)

r = results[0]
print('image:', sample.name)
print('boxes:', 0 if r.boxes is None else len(r.boxes))
if r.keypoints is not None:
    print('keypoints tensor:', tuple(r.keypoints.data.shape))

saved = Path(r.save_dir) / sample.name
if not saved.exists():
    # ultralytics may write jpg
    cands = list(Path(r.save_dir).glob(sample.stem + '.*'))
    saved = cands[0] if cands else saved
print('saved:', saved)
display(Image(filename=str(saved)))


## 5. Optional: short retrain smoke test

Runs a tiny fine-tune to verify the data pipeline. For a real training run, raise `epochs` (e.g. 50–100+) and watch Colab disconnect limits.


In [ ]:
from pathlib import Path
from ultralytics import YOLO

ROOT = Path('/content/dl')
data_yaml = ROOT / 'data/data.yaml'
weights = ROOT / 'weights/cassette_lower_epoch173.pt'

model = YOLO(str(weights))
train_results = model.train(
    data=str(data_yaml),
    epochs=3,
    imgsz=1280,
    batch=4,
    device=0,
    project=str(ROOT / 'runs/colab_train'),
    name='smoke',
    exist_ok=True,
    plots=True,
)
print(train_results)


## Tips

- Persist work: copy `runs/` to Google Drive, or `git` push new weights to a private branch.
- Free Colab may throttle long 1280 training; drop `imgsz` to 640 for faster experiments, then train full size later.
- Keypoints: `front_lower_left`, `front_lower_right` (see `data/data.yaml`).
